[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/13_gpt2_block.ipynb)

# 🔴 Hard: GPT-2 Transformer Block

Implement a full **GPT-2 style Transformer block** — combining everything you've learned.

### Architecture (Pre-Norm)
```
x = x + causal_self_attention(ln1(x))
x = x + mlp(ln2(x))
```

### Signature
```python
class GPT2Block(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor) -> torch.Tensor: ...
```

### Requirements
- Inherit from `nn.Module`
- `self.ln1`, `self.ln2`: `nn.LayerNorm(d_model)`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` for attention
- `self.mlp`: `nn.Sequential(Linear(d, 4d), GELU(), Linear(4d, d))`
- Attention must be **causal** (mask future positions)
- Pre-norm architecture (LayerNorm *before* attention and MLP)
- Residual connections around both attention and MLP

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [5]:

class GPT2Block(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_model = d_model
        self.head_dim = d_model // num_heads

        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)

        layers_mlp = [nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)]
        self.mlp = nn.Sequential(*layers_mlp)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)


    def forward(self, x):
        # Pre-norm + causal attention + MLP with residuals
        B, S_q, _ = x.shape
        S_k = S_q

        x_norm1 = self.ln1(x)

        def causal_attention(): #with x_norm1

          Q = self.W_q(x_norm1).reshape((B, S_q, self.num_heads, self.head_dim)).transpose(1, 2) #(B, num_heads, S_q, head_dim)
          K = self.W_k(x_norm1).reshape((B, S_k, self.num_heads, self.head_dim)).transpose(1, 2) #(B, num_heads, S_k, head_dim)
          V = self.W_v(x_norm1).reshape((B, S_k, self.num_heads, self.head_dim)).transpose(1, 2) #(B, num_heads, S_k, head_dim)

          d_k = K.shape[-1]
          raw_weights = (Q @ K.transpose(-1, -2)) / (d_k ** 0.5)

          mask = torch.tril(torch.ones((S_q, S_k))) == 0
          raw_weights = torch.masked_fill(raw_weights, mask, float("-inf"))

          att_weights = torch.softmax(raw_weights, dim = -1) #(B, num_heads, S_q, S_k)

          hidden = att_weights @ V #(B, num_heads, S_q, head_dim)
          hidden = hidden.transpose(1, 2).reshape((B, S_q, self.d_model)) #(B, S_q, d_model)

          return self.W_o(hidden)

        att_out = x + causal_attention()

        x_norm2 = self.ln2(att_out)

        def mlp(): #with x_norm2
          return self.mlp(x_norm2)

        mlp_out = att_out + mlp()

        return mlp_out



In [6]:
# 🧪 Debug
torch.manual_seed(0)
block = GPT2Block(d_model=64, num_heads=4)
x = torch.randn(2, 8, 64)
out = block(x)
print("Output shape:", out.shape)           # (2, 8, 64)
print("Is nn.Module?", isinstance(block, nn.Module))
print("Params:", sum(p.numel() for p in block.parameters()))

Output shape: torch.Size([2, 8, 64])
Is nn.Module? True
Params: 49984


In [7]:
from torch_judge import check
check('gpt2_block')


🧪 Testing: GPT-2 Transformer Block (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.6ms)
  ✅ [2/5] Has LayerNorm (pre-norm architecture) (1.3ms)
  ✅ [3/5] MLP has 4x expansion with GELU (1.2ms)
  ✅ [4/5] Causal masking — future doesn't affect past (26.9ms)
  ✅ [5/5] Gradient flow to all parameters (34.9ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (69.9ms total)
  Progress saved. Run status() to see your dashboard.

